# Flow Matching Defense Deep Dive (5 Attack + 5 Benign)

This notebook visualizes the full flow in the query defense:
- Run a tiny PRADA attack with Flow Matching defense (5 queries)
- Run a tiny benign baseline with the same defense (5 queries)
- Show the exact query images sent to the model
- Trace reverse-flow dynamics step-by-step and visualize Jacobian correction
- Plot OOD decision boundaries with log-likelihood threshold

## Assumptions and Trade-offs

Assumptions in this notebook:
- A trained target checkpoint already exists at TARGET_CHECKPOINT_DIR.
- The flow-matching checkpoint exists at FM_CHECKPOINT.
- Tiny analysis is constrained to 5 attack + 5 benign samples.

Trade-offs made intentionally:
- The new trace hook uses a step-wise approximation for Jacobian correction by default (Hutchinson estimator) so the visualization stays tractable.
- Solver log-likelihood is still reported as the reference decision metric.
- PRADA hyperparameters are minimized for speed, not attack strength.
- PRADA settings are tuned for a tiny, reproducible 5-query deep dive, not extraction quality.

In [ ]:
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.decomposition import PCA
from umap import UMAP

project_root = Path.cwd().resolve()
while not (project_root / "pyproject.toml").exists() and project_root != project_root.parent:
    project_root = project_root.parent
sys.path.insert(0, str(project_root / "src"))

from flowguard.defenses.query.flow_matching import FlowMatchingQueryDefense
from flowguard.experiments.factory import build_experiment_spec
from flowguard.experiments.spec import AttackKind
from flowguard.orchestration import run_experiment

NATURE = {
    "paper": "#F4F1E8",
    "ink": "#2E2A26",
    "moss": "#607D3B",
    "forest": "#355E3B",
    "clay": "#B5654A",
    "bark": "#6D4C41",
    "sky": "#5C7FA3",
    "sand": "#D8C8A8",
    "stone": "#7D8A91",
}


def apply_nature_style() -> None:
    plt.rcParams.update(
        {
            "figure.facecolor": NATURE["paper"],
            "axes.facecolor": NATURE["paper"],
            "savefig.facecolor": NATURE["paper"],
            "axes.edgecolor": NATURE["ink"],
            "axes.labelcolor": NATURE["ink"],
            "text.color": NATURE["ink"],
            "xtick.color": NATURE["ink"],
            "ytick.color": NATURE["ink"],
            "grid.color": "#C7BCA9",
            "grid.alpha": 0.35,
            "axes.grid": True,
            "axes.spines.top": False,
            "axes.spines.right": False,
            "font.family": "DejaVu Serif",
            "figure.dpi": 150,
        }
    )


apply_nature_style()
print(f"Project root: {project_root}")

In [ ]:
RUNS_DIR = project_root / "runs" / "notebook"
DEEP_DIVE_DIR = RUNS_DIR / "flow_matching_deep_dive"
DEEP_DIVE_DIR.mkdir(parents=True, exist_ok=True)

FM_CHECKPOINT = project_root / "runs" / "flow_matching" / "cifar10_notebook" / "checkpoint_latest.pt"
TARGET_CHECKPOINT_DIR = (
    project_root
    / "runs"
    / "notebook"
    / "training-victim-cifar10-vgg16_bn-nodefense"
    / "target_model"
)

QUERY_BUDGET = 20
SEED_SAMPLES = 5
PRADA_DUPLICATION_ROUNDS = 1
LIKELIHOOD_THRESHOLD = 5000.0
TRACE_STEPS = 120
TRACE_DIVERGENCE_MODE = "hutchinson"
TRACE_HUTCHINSON_SAMPLES = 1
ATTACK_DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

CIFAR_MEAN = torch.tensor([0.4914, 0.4822, 0.4465], dtype=torch.float32).view(3, 1, 1)
CIFAR_STD = torch.tensor([0.2023, 0.1994, 0.2010], dtype=torch.float32).view(3, 1, 1)


def assert_preflight() -> None:
    missing: list[Path] = []
    if not FM_CHECKPOINT.exists():
        missing.append(FM_CHECKPOINT)
    if not TARGET_CHECKPOINT_DIR.exists():
        missing.append(TARGET_CHECKPOINT_DIR)
    if missing:
        missing_lines = "\n".join(f"- {path}" for path in missing)
        raise FileNotFoundError(
            "Missing required artifact(s). Update the notebook paths before running experiments:\n"
            f"{missing_lines}"
        )


assert_preflight()
print(f"FM checkpoint: {FM_CHECKPOINT}")
print(f"Target checkpoint: {TARGET_CHECKPOINT_DIR}")
print(f"Output dir: {DEEP_DIVE_DIR}")
print(f"PRADA duplication rounds: {PRADA_DUPLICATION_ROUNDS}")

In [ ]:
def build_flow_matching_parameters() -> dict[str, Any]:
    return {
        "fm_checkpoint_path": str(FM_CHECKPOINT),
        "dataset_name": "CIFAR10",
        "device": ATTACK_DEVICE,
        "likelihood_threshold": LIKELIHOOD_THRESHOLD,
        "step_size": 0.05,
        "method": "midpoint",
        "atol": 1e-5,
        "rtol": 1e-5,
        "exact_divergence": False,
        "inputs_normalized": True,
        "audit_only": True,
    }


def build_common_spec_kwargs() -> dict[str, Any]:
    return {
        "dataset": "CIFAR10",
        "target_architecture": "vgg16_bn",
        "substitute_architecture": "vgg16_bn",
        "query_dataset": "CIFAR10",
        "query_budget": QUERY_BUDGET,
        "query_transfer_set_size": None,
        "target_checkpoint_dir": str(TARGET_CHECKPOINT_DIR),
        "target_device": ATTACK_DEVICE,
        "substitute_device": ATTACK_DEVICE,
        "distributed": False,
        "num_workers": 1,
        "num_clients": 1,
        "attack_batch_size": 1,
        "training_batch_size": 16,
        "epochs": 1,
        "verbose": True,
    }


def run_tiny_prada_with_flow_matching() -> tuple[Any, Any]:
    prada_extra = {
        "initial_seed_size": SEED_SAMPLES,
        "duplication_rounds": PRADA_DUPLICATION_ROUNDS,
        "hyperparameter_search_budget": 0,
        "verbose": True,
    }
    spec = build_experiment_spec(
        name="fm-deep-dive-prada-5q",
        attack_kind=AttackKind.PRADA,
        attack_mode="prada",
        query_defense="flow_matching",
        query_defense_parameters=build_flow_matching_parameters(),
        attack_extra=prada_extra,
        **build_common_spec_kwargs(),
    )
    result = run_experiment(spec, output_dir=DEEP_DIVE_DIR / spec.name)
    return spec, result


def run_tiny_benign_with_flow_matching() -> tuple[Any, Any]:
    spec = build_experiment_spec(
        name="fm-deep-dive-benign-5q",
        attack_kind=AttackKind.TRANSFER_SET,
        attack_mode="naive",
        query_defense="flow_matching",
        query_defense_parameters=build_flow_matching_parameters(),
        **build_common_spec_kwargs(),
    )
    result = run_experiment(spec, output_dir=DEEP_DIVE_DIR / spec.name)
    return spec, result


def _ensure_chw(image: torch.Tensor) -> torch.Tensor:
    tensor = image.detach().cpu().float()
    if tensor.ndim == 2:
        tensor = tensor.unsqueeze(0)
    elif tensor.ndim == 3 and tensor.shape[0] not in {1, 3} and tensor.shape[-1] in {1, 3}:
        tensor = tensor.permute(2, 0, 1)
    return tensor.contiguous()


def _to_model_input(image: torch.Tensor) -> torch.Tensor:
    tensor = _ensure_chw(image)
    if tensor.min().item() >= 0.0 and tensor.max().item() > 5.0:
        tensor = tensor / 255.0
        tensor = (tensor - CIFAR_MEAN) / CIFAR_STD
    return tensor


def _load_transfer_like_records(path: Path) -> list[dict[str, Any]]:
    payload = torch.load(path, map_location="cpu", weights_only=False)
    if not isinstance(payload, list):
        raise TypeError(f"Expected a list payload in {path}, got {type(payload)}")

    records: list[dict[str, Any]] = []
    for index, entry in enumerate(payload):
        if isinstance(entry, dict):
            query_x = torch.as_tensor(entry.get("query_x", entry.get("x")), dtype=torch.float32)
            original_x = torch.as_tensor(entry.get("original_x", query_x), dtype=torch.float32)
            query_index = int(entry.get("query_index", index))
        else:
            query_x = torch.as_tensor(entry[0], dtype=torch.float32)
            original_x = query_x.clone()
            query_index = index

        records.append(
            {
                "query_x": _to_model_input(query_x),
                "original_x": _to_model_input(original_x),
                "query_index": query_index,
            }
        )

    records.sort(key=lambda item: item["query_index"])
    return records


def _extract_scores_and_flags(result: Any) -> tuple[list[float], list[bool]]:
    scores: list[float] = []
    flags: list[bool] = []
    if result.query_summary.history is None:
        return scores, flags

    for record in result.query_summary.history.records:
        raw_scores = record.metadata.get("queries_blocked", [])
        raw_flags = record.metadata.get("flow_matching_blocked", [])
        for score, flag in zip(raw_scores, raw_flags):
            scores.append(float(score))
            flags.append(bool(flag))
    return scores, flags


def to_display_image_target(query_tensor: torch.Tensor) -> np.ndarray:
    image = _ensure_chw(query_tensor)
    if image.min().item() < -0.25 or image.max().item() > 1.25:
        image = image * CIFAR_STD + CIFAR_MEAN
    elif image.min().item() >= 0.0 and image.max().item() > 5.0:
        image = image / 255.0
    image = image.clamp(0.0, 1.0)
    if image.shape[0] == 1:
        image = image.repeat(3, 1, 1)
    return np.transpose(image.numpy(), (1, 2, 0))


def to_display_image_fm(fm_tensor: torch.Tensor) -> np.ndarray:
    image = _ensure_chw(fm_tensor)
    image = ((image + 1.0) * 0.5).clamp(0.0, 1.0)
    if image.shape[0] == 1:
        image = image.repeat(3, 1, 1)
    return np.transpose(image.numpy(), (1, 2, 0))

In [ ]:
FORCE_RERUN_EXPERIMENTS = True

if FORCE_RERUN_EXPERIMENTS or "prada_result" not in globals() or "benign_result" not in globals():
    prada_spec, prada_result = run_tiny_prada_with_flow_matching()
    benign_spec, benign_result = run_tiny_benign_with_flow_matching()
else:
    print("Reusing existing prada_result and benign_result from kernel state.")

print("PRADA run:")
print(f"  total queries: {prada_result.query_summary.total_queries}")
print(f"  output dir: {prada_result.attack_summary.output_dir}")
print("Benign run:")
print(f"  total queries: {benign_result.query_summary.total_queries}")
print(f"  output dir: {benign_result.attack_summary.output_dir}")

prada_meta = prada_result.attack_summary.metadata
prada_records_path = Path(prada_meta.get("visualization_records_path", prada_meta.get("transferset_path", "")))
if not prada_records_path.exists():
    raise FileNotFoundError(f"Could not find PRADA records at {prada_records_path}")

benign_records_path = Path(benign_result.attack_summary.metadata.get("transferset_path", ""))
if not benign_records_path.exists():
    raise FileNotFoundError(f"Could not find benign records at {benign_records_path}")

attack_records_all = _load_transfer_like_records(prada_records_path)
benign_records_all = _load_transfer_like_records(benign_records_path)

attack_scores_all, attack_flags_all = _extract_scores_and_flags(prada_result)
benign_scores_all, benign_flags_all = _extract_scores_and_flags(benign_result)

attack_count = min(
    QUERY_BUDGET,
    len(attack_records_all),
    len(attack_scores_all),
    len(attack_flags_all),
)
benign_count = min(
    QUERY_BUDGET,
    len(benign_records_all),
    len(benign_scores_all),
    len(benign_flags_all),
)
ANALYSIS_QUERY_COUNT = min(attack_count, benign_count)

if ANALYSIS_QUERY_COUNT <= 0:
    raise RuntimeError("No aligned attack/benign records available for analysis.")

if ANALYSIS_QUERY_COUNT < QUERY_BUDGET:
    print(
        f"Using {ANALYSIS_QUERY_COUNT} paired queries per source "
        f"(requested {QUERY_BUDGET}, attack available {attack_count}, benign available {benign_count})."
    )

attack_records = attack_records_all[:ANALYSIS_QUERY_COUNT]
benign_records = benign_records_all[:ANALYSIS_QUERY_COUNT]
attack_scores = attack_scores_all[:ANALYSIS_QUERY_COUNT]
attack_flags = attack_flags_all[:ANALYSIS_QUERY_COUNT]
benign_scores = benign_scores_all[:ANALYSIS_QUERY_COUNT]
benign_flags = benign_flags_all[:ANALYSIS_QUERY_COUNT]
ATTACK_OFFSET = ANALYSIS_QUERY_COUNT

combined_records: list[dict[str, Any]] = []
for index in range(ANALYSIS_QUERY_COUNT):
    combined_records.append(
        {
            "source": "benign",
            "query_id": index,
            "query_x": benign_records[index]["query_x"],
            "original_x": benign_records[index]["original_x"],
            "history_score": benign_scores[index],
            "history_blocked": benign_flags[index],
        }
    )
for index in range(ANALYSIS_QUERY_COUNT):
    combined_records.append(
        {
            "source": "attack",
            "query_id": ATTACK_OFFSET + index,
            "query_x": attack_records[index]["query_x"],
            "original_x": attack_records[index]["original_x"],
            "history_score": attack_scores[index],
            "history_blocked": attack_flags[index],
        }
    )

print(f"\nCombined {2 * ANALYSIS_QUERY_COUNT}-query table:")
for item in combined_records:
    print(
        f"id={item['query_id']:02d} source={item['source']:<6} "
        f"score={item['history_score']:.2f} blocked={item['history_blocked']}"
    )

In [ ]:
fig, axes = plt.subplots(
    2,
    ANALYSIS_QUERY_COUNT,
    figsize=(3.0 * ANALYSIS_QUERY_COUNT, 6.2),
    constrained_layout=True,
    squeeze=False,
)

benign_subset = [item for item in combined_records if item["source"] == "benign"]
attack_subset = [item for item in combined_records if item["source"] == "attack"]

for col, item in enumerate(benign_subset):
    axis = axes[0, col]
    axis.imshow(to_display_image_target(item["query_x"]))
    axis.set_title(
        f"Benign #{col + 1}\nscore={item['history_score']:.1f}",
        fontsize=10,
        color=NATURE["forest"],
    )
    border_color = NATURE["clay"] if item["history_blocked"] else NATURE["forest"]
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2.0)
        spine.set_edgecolor(border_color)
    axis.set_xticks([])
    axis.set_yticks([])

for col, item in enumerate(attack_subset):
    axis = axes[1, col]
    axis.imshow(to_display_image_target(item["query_x"]))
    axis.set_title(
        f"Attack #{col + 1}\nscore={item['history_score']:.1f}",
        fontsize=10,
        color=NATURE["clay"],
    )
    border_color = NATURE["clay"] if item["history_blocked"] else NATURE["forest"]
    for spine in axis.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(2.0)
        spine.set_edgecolor(border_color)
    axis.set_xticks([])
    axis.set_yticks([])

axes[0, 0].set_ylabel("Benign\nqueries", fontsize=12)
axes[1, 0].set_ylabel("PRADA\nqueries", fontsize=12)
fig.suptitle(
    f"Images used to query the target model ({ANALYSIS_QUERY_COUNT} benign + {ANALYSIS_QUERY_COUNT} attack)",
    fontsize=15,
)
plt.show()

In [ ]:
flow_defense = FlowMatchingQueryDefense(
    fm_checkpoint_path=str(FM_CHECKPOINT),
    dataset_name="CIFAR10",
    device=ATTACK_DEVICE,
    likelihood_threshold=LIKELIHOOD_THRESHOLD,
    step_size=0.05,
    method="midpoint",
    atol=1e-5,
    rtol=1e-5,
    exact_divergence=False,
    inputs_normalized=True,
    audit_only=True,
)

combined_batch = torch.stack([item["query_x"] for item in combined_records], dim=0).float()
trace = flow_defense.trace_likelihood_path(
    combined_batch,
    num_steps=TRACE_STEPS,
    divergence_mode=TRACE_DIVERGENCE_MODE,
    hutchinson_samples=TRACE_HUTCHINSON_SAMPLES,
    include_solver_reference=True,
)

solver_scores = trace["solver_log_likelihood"]
if solver_scores is None:
    solver_scores = trace["trace_log_likelihood"]
solver_flags = solver_scores > LIKELIHOOD_THRESHOLD

trace_x1 = trace["prepared_inputs"]
trace_x0 = trace["trace_x_0"]
solver_x0 = trace["solver_x_0"]


def _delta_stats(a: torch.Tensor, b: torch.Tensor) -> dict[str, torch.Tensor]:
    diff = (a - b).float().flatten(start_dim=1)
    return {
        "mean_abs_per_query": diff.abs().mean(dim=1),
        "max_abs_per_query": diff.abs().max(dim=1).values,
        "l2_per_query": torch.linalg.vector_norm(diff, dim=1),
    }


def _print_delta_metrics(name: str, stats: dict[str, torch.Tensor]) -> None:
    mean_abs = stats["mean_abs_per_query"]
    max_abs = stats["max_abs_per_query"]
    l2 = stats["l2_per_query"]
    print(f"\n{name}:")
    print(f"  avg mean(|delta|): {float(mean_abs.mean().item()):.6f}")
    print(f"  avg max(|delta|):  {float(max_abs.mean().item()):.6f}")
    print(f"  avg L2(delta):     {float(l2.mean().item()):.6f}")
    print("  per-query L2:", ", ".join(f"{value:.6f}" for value in l2.tolist()))


delta_trace_vs_x1 = _delta_stats(trace_x0, trace_x1)
delta_solver_vs_x1 = None
delta_solver_vs_trace = None
if solver_x0 is not None:
    delta_solver_vs_x1 = _delta_stats(solver_x0, trace_x1)
    delta_solver_vs_trace = _delta_stats(solver_x0, trace_x0)

for idx, item in enumerate(combined_records):
    item["solver_score"] = float(solver_scores[idx].item())
    item["solver_blocked"] = bool(solver_flags[idx].item())
    item["trace_log_p0"] = float(trace["trace_log_p0"][idx].item())
    item["trace_correction"] = float(trace["jacobian_correction_total"][idx].item())
    item["trace_score"] = float(trace["trace_log_likelihood"][idx].item())
    item["delta_trace_vs_x1_l2"] = float(delta_trace_vs_x1["l2_per_query"][idx].item())
    if delta_solver_vs_x1 is not None:
        item["delta_solver_vs_x1_l2"] = float(delta_solver_vs_x1["l2_per_query"][idx].item())
    if delta_solver_vs_trace is not None:
        item["delta_solver_vs_trace_l2"] = float(delta_solver_vs_trace["l2_per_query"][idx].item())

residual = trace["trace_log_likelihood"] - (
    trace["trace_log_p0"] + trace["jacobian_correction_total"]
)
print(f"Trace decomposition residual max abs: {float(residual.abs().max().item()):.6f}")

print("\nScore comparison table:")
for item in combined_records:
    print(
        f"id={item['query_id']:02d} source={item['source']:<6} "
        f"history={item['history_score']:.2f} solver={item['solver_score']:.2f} "
        f"trace={item['trace_score']:.2f} blocked={item['solver_blocked']}"
    )

print("\nDelta metrics:")
_print_delta_metrics("Trace x0 vs x1", delta_trace_vs_x1)
if delta_solver_vs_x1 is not None and delta_solver_vs_trace is not None:
    _print_delta_metrics("Solver x0 vs x1", delta_solver_vs_x1)
    _print_delta_metrics("Solver x0 vs Trace x0", delta_solver_vs_trace)
else:
    print("Solver x0 is not available (include_solver_reference=False).")

In [ ]:
candidate_indices = [0, 1, ATTACK_OFFSET, ATTACK_OFFSET + 1]
selected_indices = [idx for idx in candidate_indices if 0 <= idx < len(combined_records)]
if not selected_indices:
    raise RuntimeError("No query indices available for x1/x0 visualization.")

prepared_inputs = trace["prepared_inputs"]
trace_x0 = trace["trace_x_0"]

fig, axes = plt.subplots(len(selected_indices), 2, figsize=(10.5, 2.8 * len(selected_indices)), constrained_layout=True)
if len(selected_indices) == 1:
    axes = np.expand_dims(axes, axis=0)

for row, idx in enumerate(selected_indices):
    item = combined_records[idx]
    # axes[row, 0].imshow(to_display_image_target(item["query_x"]))
    # axes[row, 0].set_title(f"Target query (id={item['query_id']})", fontsize=10)

    axes[row, 0].imshow(to_display_image_fm(prepared_inputs[idx]))
    axes[row, 0].set_title("Flow input x1", fontsize=10)

    axes[row, 1].imshow(to_display_image_fm(trace_x0[idx]))
    axes[row, 1].set_title("Reverse-flow endpoint x0", fontsize=10)

    for col in range(2):
        axes[row, col].set_xticks([])
        axes[row, col].set_yticks([])

fig.suptitle("How queries are transformed inside Flow Matching defense", fontsize=15)
plt.show()

In [ ]:
trajectory = trace["trajectory"]
num_queries, num_time_points = trajectory.shape[0], trajectory.shape[1]
flat_states = trajectory.reshape(num_queries * num_time_points, -1).numpy()

reference_count = 250
gaussian_reference = torch.randn(reference_count, *trajectory.shape[2:])
flat_reference = gaussian_reference.reshape(reference_count, -1).numpy()


def build_trajectory_embedding(method: str) -> tuple[np.ndarray, np.ndarray]:
    stacked = np.vstack([flat_states, flat_reference])
    if method == "pca":
        reducer = PCA(n_components=2, random_state=0)
    elif method == "umap":
        reducer = UMAP(
            n_components=2,
            random_state=0,
            n_neighbors=max(2, min(15, stacked.shape[0] - 1)),
            min_dist=0.25,
        )
    else:
        raise ValueError(f"Unsupported method: {method}")

    embedding = reducer.fit_transform(stacked)
    state_embedding = embedding[: num_queries * num_time_points].reshape(num_queries, num_time_points, 2)
    reference_embedding = embedding[num_queries * num_time_points :]
    return state_embedding, reference_embedding


pca_embedding, pca_reference = build_trajectory_embedding("pca")
umap_embedding, umap_reference = build_trajectory_embedding("umap")

fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.2), constrained_layout=True)
for axis, method_name, state_embedding, reference_embedding in [
    (axes[0], "PCA", pca_embedding, pca_reference),
    (axes[1], "UMAP", umap_embedding, umap_reference),
]:
    axis.scatter(
        reference_embedding[:, 0],
        reference_embedding[:, 1],
        s=10,
        color=NATURE["stone"],
        alpha=0.20,
        label="Standard Gaussian reference",
    )

    arrow_stride = max(1, (num_time_points - 1) // 6)
    for index, item in enumerate(combined_records):
        color = NATURE["forest"] if item["source"] == "benign" else NATURE["clay"]
        style = "-" if item["source"] == "benign" else "--"
        path = state_embedding[index]
        axis.plot(path[:, 0], path[:, 1], linestyle=style, linewidth=1.7, color=color, alpha=0.9)
        axis.scatter(path[0, 0], path[0, 1], s=34, marker="o", color=color, edgecolors=NATURE["ink"], linewidths=0.4)
        axis.scatter(path[-1, 0], path[-1, 1], s=48, marker="X", color=color, edgecolors=NATURE["ink"], linewidths=0.4)

        for step in range(0, num_time_points - 1, arrow_stride):
            delta = path[step + 1] - path[step]
            axis.arrow(
                path[step, 0],
                path[step, 1],
                delta[0],
                delta[1],
                width=0.0007,
                head_width=0.06,
                length_includes_head=True,
                color=color,
                alpha=0.45,
            )

    axis.plot([], [], color=NATURE["forest"], linestyle="-", label="Benign trajectory")
    axis.plot([], [], color=NATURE["clay"], linestyle="--", label="PRADA trajectory")
    axis.scatter([], [], marker="o", s=34, color=NATURE["sand"], edgecolors=NATURE["ink"], linewidths=0.4, label="t=1 (query)")
    axis.scatter([], [], marker="X", s=48, color=NATURE["bark"], edgecolors=NATURE["ink"], linewidths=0.4, label="t=0 (x0)")
    axis.set_title(f"{method_name}: reverse-flow movement over vector field", fontsize=12)
    axis.set_xlabel("Component 1")
    axis.set_ylabel("Component 2")
    axis.legend(loc="best", fontsize=8)

fig.suptitle("How attack/benign queries move through the Flow Matching reverse dynamics", fontsize=15)
plt.show()

In [ ]:
prepared_flat = trace["prepared_inputs"].flatten().numpy()
trace_x0_flat = trace["trace_x_0"].flatten().numpy()
normal_reference = np.random.default_rng(0).normal(loc=0.0, scale=1.0, size=trace_x0_flat.shape[0])

fig, ax = plt.subplots(figsize=(10.2, 4.6), constrained_layout=True)
ax.hist(prepared_flat, bins=70, density=True, alpha=0.42, color=NATURE["sky"], label="x1 (after FM preprocessing)")
ax.hist(trace_x0_flat, bins=70, density=True, alpha=0.42, color=NATURE["moss"], label="x0 (reverse-flow endpoint)")
ax.hist(normal_reference, bins=70, density=True, alpha=0.22, color=NATURE["stone"], label="N(0, 1) reference")
ax.set_title("Where queries land after reverse flow: x0 vs standard Gaussian", fontsize=14)
ax.set_xlabel("Value")
ax.set_ylabel("Density")
ax.legend(loc="upper left")
plt.show()

query_ids = np.array([item["query_id"] for item in combined_records])
history_scores_np = np.array([item["history_score"] for item in combined_records])
solver_scores_np = np.array([item["solver_score"] for item in combined_records])
log_p0_np = np.array([item["trace_log_p0"] for item in combined_records])
correction_np = np.array([item["trace_correction"] for item in combined_records])

point_colors = [NATURE["forest"] if item["source"] == "benign" else NATURE["clay"] for item in combined_records]
point_markers = ["o" if not item["solver_blocked"] else "X" for item in combined_records]

fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.6), constrained_layout=True)

for idx in range(len(combined_records)):
    axes[0].scatter(
        query_ids[idx],
        solver_scores_np[idx],
        color=point_colors[idx],
        marker=point_markers[idx],
        s=90,
        edgecolors=NATURE["ink"],
        linewidths=0.35,
    )
axes[0].plot(query_ids, history_scores_np, color=NATURE["sky"], linewidth=1.2, alpha=0.85, label="History score")
axes[0].plot(query_ids, solver_scores_np, color=NATURE["bark"], linewidth=1.2, alpha=0.85, label="Solver score")
axes[0].axhline(LIKELIHOOD_THRESHOLD, color=NATURE["clay"], linestyle="--", linewidth=1.4, label="Decision threshold")
axes[0].set_title("OOD decision from log-likelihood", fontsize=13)
axes[0].set_xlabel("Query id")
axes[0].set_ylabel("Log-likelihood")
axes[0].legend(loc="best")

axes[1].bar(query_ids, log_p0_np, color=NATURE["moss"], alpha=0.86, label="log p0(x0)")
axes[1].bar(query_ids, correction_np, bottom=log_p0_np, color=NATURE["sand"], alpha=0.94, label="Jacobian correction")
axes[1].scatter(query_ids, solver_scores_np, color=NATURE["ink"], s=30, label="Final score")
axes[1].set_title("Score decomposition: log p(x1) = log p0(x0) + correction", fontsize=13)
axes[1].set_xlabel("Query id")
axes[1].set_ylabel("Contribution")
axes[1].legend(loc="best")

plt.show()

time_grid = trace["time_grid"].numpy()
correction_cumulative = trace["jacobian_correction_cumulative"].numpy()
step_times = time_grid[1:]

benign_index = 0
attack_index = ATTACK_OFFSET
if attack_index >= len(combined_records):
    raise RuntimeError("Attack index is out of range for Jacobian correction plot.")

fig, ax = plt.subplots(figsize=(10.0, 4.8), constrained_layout=True)
ax.plot(
    step_times,
    correction_cumulative[benign_index],
    color=NATURE["forest"],
    linewidth=2.0,
    label="Benign cumulative correction",
)
ax.plot(
    step_times,
    correction_cumulative[attack_index],
    color=NATURE["clay"],
    linewidth=2.0,
    label="PRADA cumulative correction",
)
ax.set_title("Step-wise Jacobian correction along reverse ODE path", fontsize=14)
ax.set_xlabel("Time t (reverse integration from 1 to 0)")
ax.set_ylabel("Cumulative correction")
ax.legend(loc="best")
ax.invert_xaxis()
plt.show()